In [23]:
import torch
import torch.nn as nn
import torchvision
import pandas as pd
from PIL import Image
import numpy as np
# We have kept all imports here.

In [24]:

if torch.backends.mps.is_available():
    device = torch.device("mps")   # Apple GPU (if available)
else:
    device = torch.device("cpu")   # fallback for Mac CPU

print("Using device:", device)

Using device: mps


In [25]:


train_df = pd.read_csv("train_split.csv")
val_df   = pd.read_csv("val_split.csv")
test_df  = pd.read_csv("test_split.csv")


train_df.head()
train_df.columns
train_df['label'].value_counts

# verified the csv file structure

<bound method IndexOpsMixin.value_counts of 0       1
1       0
2       0
3       1
4       4
       ..
8007    1
8008    1
8009    1
8010    1
8011    1
Name: label, Length: 8012, dtype: int64>

In [26]:
from PIL import Image

DATA_ROOT = "/Users/harshnayak/ham10000_project/data"



img_path = f"{DATA_ROOT}/" + train_df.iloc[0]['path'].replace("HAM10000/", "")

print(img_path)

#merged the file path mentioned in the csv to the actual image path and opened the image

/Users/harshnayak/ham10000_project/data/HAM10000_images_part_2/ISIC_0031481.jpg


In [27]:
img = Image.open(img_path)
img.show()

# checked the image plotted it and made it visible

In [28]:
labels = train_df['label'].values
class_counts = np.bincount(labels)
class_weights = 1.0 / class_counts
class_weights = class_weights / class_weights.sum()
class_weights = torch.tensor(class_weights, dtype=torch.float)

# here we do the class weight handeline this is done because some classes have a high number in the dataset and we have a high chance that the model memorizes them and just predicts that 

In [29]:
print(class_weights)
print(torch.isnan(class_weights).any())
print(torch.isinf(class_weights).any())

tensor([0.0396, 0.0066, 0.0859, 0.1347, 0.0401, 0.3836, 0.3095])
tensor(False)
tensor(False)


Next Stage is to create data loaders, they convert the csv to efficient net inputs. 

In [30]:
from PIL import Image
from torch.utils.data import Dataset

class HAMDataset(Dataset):
    def __init__(self, df, root_dir, transform=None):
        self.df = df
        self.root_dir = root_dir   # <-- DATA_ROOT stored here
        self.transform = transform

    def __len__(self):
        return len(self.df)

    def __getitem__(self, idx):
        path = self.df.iloc[idx]['path']
        path = path.replace("HAM10000/", "")
        img_path = f"{self.root_dir}/{path}"

        # 🔍 DEBUG: check if file exists
        import os
        if not os.path.exists(img_path):
            print("BROKEN PATH:", img_path)
            raise FileNotFoundError(img_path)

        image = Image.open(img_path).convert("RGB")
        label = self.df.iloc[idx]['label']

        if self.transform:
            image = self.transform(image)

        return image, label



In [ ]:
   def __getitem__(self, idx):
        path = self.df.iloc[idx]['path']

        # remove wrong prefix (your working logic)
        path = path.replace("HAM10000/", "")

        # use root_dir properly
        img_path = f"{self.root_dir}/{path}"

        image = Image.open(img_path).convert("RGB")
        label = self.df.iloc[idx]['label']

        if self.transform:
            image = self.transform(image)

        return image, label

Next stage is transforming images, tensor conversion to fit properly to in the dataset. 

In [31]:
from torchvision import transforms

In [32]:
train_transform = transforms.Compose([
    transforms.Resize((224, 224)),   # EfficientNet input size
    transforms.RandomHorizontalFlip(),  # augmentation
    transforms.RandomRotation(10),      # augmentation
    transforms.ToTensor(),              # PIL → tensor
    transforms.Normalize(
        mean=[0.485, 0.456, 0.406],     # ImageNet stats
        std=[0.229, 0.224, 0.225]
    )
])

In [33]:

train_dataset = HAMDataset(train_df, DATA_ROOT, transform=train_transform)
val_dataset = HAMDataset(val_df, DATA_ROOT, transform=train_transform)
test_dataset = HAMDataset(test_df, DATA_ROOT, transform=train_transform)

In [34]:
img, label = train_dataset[0]

print(type(img))
print(img.shape)

<class 'torch.Tensor'>
torch.Size([3, 224, 224])


next step is data loader 

In [35]:
from torch.utils.data import DataLoader

train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=32, shuffle=False)

In [36]:
images, labels = next(iter(train_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [37]:
images, labels = next(iter(test_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


In [38]:
images, labels = next(iter(val_loader))

print(images.shape)
print(labels.shape)

torch.Size([32, 3, 224, 224])
torch.Size([32])


This step is model training step , we will start with s2 strategy here, and lets see how it goes

In [41]:
import torchvision.models as models
import torch.nn as nn

def build_s2_full_freeze():
    model = models.efficientnet_b0(weights='IMAGENET1K_V1')

    for param in model.parameters():
        param.requires_grad = False

    model.classifier[1] = nn.Linear(
        model.classifier[1].in_features,
        7
    )

    print("S2 - Full Freeze: only classification head trainable")
    return model

In [42]:
### first we define the model and weights

model = build_s2_full_freeze()
model = model.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)
import torch.optim as optim

optimizer = optim.Adam(
    filter(lambda p: p.requires_grad, model.parameters()),
    lr=1e-3
)



S2 - Full Freeze: only classification head trainable


In [43]:
class_weights = class_weights.to(device)
criterion = nn.CrossEntropyLoss(weight=class_weights)

model = model.to(device)

num_epochs = 10

for epoch in range(num_epochs):

    # =========================
    # TRAIN PHASE
    # =========================
    model.train()

    train_loss = 0
    train_correct = 0
    train_total = 0

    for images, labels in train_loader:
        images = images.to(device)
        labels = labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        train_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        train_correct += (preds == labels).sum().item()
        train_total += labels.size(0)

    avg_train_loss = train_loss / len(train_loader)
    train_acc = train_correct / train_total


    # =========================
    # VALIDATION PHASE
    # =========================
    model.eval()

    val_loss = 0
    val_correct = 0
    val_total = 0

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for images, labels in val_loader:
            images = images.to(device)
            labels = labels.to(device)

            outputs = model(images)
            loss = criterion(outputs, labels)

            val_loss += loss.item()

            _, preds = torch.max(outputs, 1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

            all_preds.extend(preds.cpu().numpy())
            all_labels.extend(labels.cpu().numpy())

    avg_val_loss = val_loss / len(val_loader)
    val_acc = val_correct / val_total


    # =========================
    # PRINT METRICS
    # =========================
    print(f"\nEpoch [{epoch+1}/{num_epochs}]")
    print(f"Train Loss: {avg_train_loss:.4f} | Train Acc: {train_acc:.4f}")
    print(f"Val   Loss: {avg_val_loss:.4f} | Val   Acc: {val_acc:.4f}")
    print("-" * 50)


Epoch [1/10]
Train Loss: 1.4658 | Train Acc: 0.5708
Val   Loss: 1.1365 | Val   Acc: 0.6164
--------------------------------------------------

Epoch [2/10]
Train Loss: 1.1627 | Train Acc: 0.6171
Val   Loss: 1.0370 | Val   Acc: 0.6663
--------------------------------------------------

Epoch [3/10]
Train Loss: 1.0672 | Train Acc: 0.6324
Val   Loss: 0.9845 | Val   Acc: 0.6454
--------------------------------------------------

Epoch [4/10]
Train Loss: 1.0388 | Train Acc: 0.6399
Val   Loss: 0.9304 | Val   Acc: 0.6733
--------------------------------------------------

Epoch [5/10]
Train Loss: 0.9634 | Train Acc: 0.6510
Val   Loss: 0.9450 | Val   Acc: 0.6603
--------------------------------------------------

Epoch [6/10]
Train Loss: 0.9655 | Train Acc: 0.6555
Val   Loss: 0.9321 | Val   Acc: 0.6803
--------------------------------------------------

Epoch [7/10]
Train Loss: 0.9396 | Train Acc: 0.6499
Val   Loss: 0.9282 | Val   Acc: 0.6583
--------------------------------------------------

In [44]:
from sklearn.metrics import (
    balanced_accuracy_score,
    f1_score,
    confusion_matrix,
    classification_report
)

In [45]:
bal_acc = balanced_accuracy_score(all_labels, all_preds)

macro_f1 = f1_score(
    all_labels,
    all_preds,
    average='macro'
)

print("Balanced Accuracy:", bal_acc)
print("Macro F1 Score:", macro_f1)

Balanced Accuracy: 0.6935480833574399
Macro F1 Score: 0.5114206732068635


In [46]:
cm = confusion_matrix(all_labels, all_preds)

print(cm)

[[ 73  10   2   8  12   3   3]
 [ 87 462  17  12  43  26  23]
 [  1   0  31  11   2   5   1]
 [  1   0   4  22   2   2   2]
 [ 20   6   9  11  61   2   1]
 [  0   1   0   1   1   9   0]
 [  0   0   1   0   0   0  13]]


In [47]:
print(classification_report(all_labels, all_preds))

              precision    recall  f1-score   support

           0       0.40      0.66      0.50       111
           1       0.96      0.69      0.80       670
           2       0.48      0.61      0.54        51
           3       0.34      0.67      0.45        33
           4       0.50      0.55      0.53       110
           5       0.19      0.75      0.31        12
           6       0.30      0.93      0.46        14

    accuracy                           0.67      1001
   macro avg       0.46      0.69      0.51      1001
weighted avg       0.79      0.67      0.70      1001



In [115]:
torch.save(model.state_dict(), "s2_efficientnet.pth")

now we go to s1 